In [ ]:
"""
SQUID Measurement Analysis — Background Correction, Averaging & Metrics
========================================================================
Loads multiple raw SQUID .dat files for a single particle batch (same
diameter), optionally subtracts the diamagnetic/paramagnetic substrate
background, averages the repeated measurements, and exports plots and a
statistical report.

Background correction strategy (when enabled):
  A linear slope is fitted independently to the positive tail (top 5% of
  field range) and negative tail (bottom 5%) where FePt is fully saturated.
  The two slopes are averaged and subtracted from the full curve, isolating
  the ferromagnetic FePt contribution from the non-magnetic SiO2 / holder.

Panel layout:
  - Top panel   : raw emu — individual curves in sweep order + mean ± 1 SD
                  (NO background correction, exactly as measured)
  - Bottom panel : background-corrected, normalised M/Msat — individual
                  curves + mean ± 1 SD

Inputs  : Raw SQUID .dat files (Quantum Design format with [Data] header).
Outputs : PNG + SVG plot, .txt statistical report, and averaged data CSV —
          all saved to a timestamped batch folder.
"""

# --- Standard library ---
import os
from datetime import datetime

# --- Third-party ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- GUI ---
import tkinter as tk
from tkinter import filedialog, simpledialog


# ===========================================================================
# 1. SETTINGS & CONSTANTS
# ===========================================================================

OUTDIR_BASE  = "squid_batch_analysis"
N_GRID       = 3000   # Points on the common interpolation grid
SAT_FRAC     = 0.05   # Outermost fraction of field used to fit background slope
STD_SMOOTH   = 61     # Moving-average window for smoothing the ±1 SD band (odd)

DENSITY_G_CM3 = 15.0  # FePt bulk density [g/cm³]
OE_TO_T       = 1e-4  # Oersted → Tesla


# ===========================================================================
# 2. PHYSICS & GEOMETRY HELPERS
# ===========================================================================

def get_fept_volume_and_mass(substrate_area_mm2, sphere_diam_um, thick_nm):
    """
    Estimate total FePt volume and mass for a hexagonal close-packed
    monolayer of spheres on the substrate (packing fraction 0.906).
    """
    sphere_r_mm = (sphere_diam_um / 2) * 1e-3
    n_spheres   = (substrate_area_mm2 * 0.906) / (2 * np.sqrt(3) * sphere_r_mm**2)
    R_cm        = (sphere_diam_um / 2) * 1e-4
    t_cm        = thick_nm * 1e-7
    vol_cap_cm3 = (2/3) * np.pi * ((R_cm + t_cm)**3 - R_cm**3)
    total_vol   = n_spheres * vol_cap_cm3
    return total_vol, total_vol * DENSITY_G_CM3


def get_metrics(h, m, mass_g):
    """
    Extract coercivity [T], squareness (Mr/Ms), and hysteresis loss [J/kg]
    from a full M(H) loop (h in Oe, m in emu).
    """
    idx_hc      = np.argmin(np.abs(m))
    hc_t        = np.abs(h[idx_hc]) * OE_TO_T
    msat        = np.max(np.abs(m))
    mr          = np.interp(0, h[::-1], m[::-1])
    mr_ms       = np.abs(mr / msat)
    area_emu_oe = np.abs(np.trapezoid(m, h))
    whyst       = (area_emu_oe * 1e-7) / (mass_g * 1e-3)
    return hc_t, mr_ms, whyst


def split_branches(h, m):
    """
    Split a full hysteresis loop into descending (H+ → H−) and ascending
    (H− → H+) branches. Averaging branches separately ensures the mean loop
    is a proper closed hysteresis curve rather than a collapsed S-curve.
    """
    idx_max, idx_min = np.argmax(h), np.argmin(h)
    if idx_max < idx_min:
        h_desc, m_desc = h[idx_max:idx_min+1], m[idx_max:idx_min+1]
        h_asc  = np.concatenate([h[idx_min:], h[:idx_max]])
        m_asc  = np.concatenate([m[idx_min:], m[:idx_max]])
    else:
        h_asc,  m_asc  = h[idx_min:idx_max+1], m[idx_min:idx_max+1]
        h_desc = np.concatenate([h[idx_max:], h[:idx_min]])
        m_desc = np.concatenate([m[idx_max:], m[:idx_min]])
    return (h_desc, m_desc), (h_asc, m_asc)


def moving_average(y, n):
    """Box-car moving average of window n (forced odd). Returns y if n < 3."""
    if n < 3:
        return y
    if n % 2 == 0:
        n += 1
    return np.convolve(np.asarray(y), np.ones(n) / n, mode="same")


def subtract_linear_background(h, m):
    """
    Remove the linear diamagnetic/paramagnetic substrate background.

    Fits the positive tail (h > 95% of h_max) and negative tail
    (h < -95% of h_max) independently — at these extreme fields FePt is
    fully saturated so remaining field-linear signal is purely from the
    substrate. The two slopes are averaged for robustness, then subtracted.
    """
    h, m      = np.asarray(h), np.asarray(m)
    h_abs_max = np.max(np.abs(h))
    mask_pos  = h >  h_abs_max * (1 - SAT_FRAC)
    mask_neg  = h < -h_abs_max * (1 - SAT_FRAC)
    slope_pos, _ = np.polyfit(h[mask_pos], m[mask_pos], 1) if mask_pos.sum() > 2 else (0, 0)
    slope_neg, _ = np.polyfit(h[mask_neg], m[mask_neg], 1) if mask_neg.sum() > 2 else (0, 0)
    slope = (slope_pos + slope_neg) / 2
    return m - slope * h, slope


# ===========================================================================
# 3. MAIN
# ===========================================================================

def main():

    # --- Sample parameters via GUI dialogs ---
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    diam  = simpledialog.askfloat("Input", "Sphere diameter (µm):", initialvalue=3.0)
    thick = simpledialog.askfloat("Input", "Cap thickness (nm):",   initialvalue=60.0)
    area  = simpledialog.askfloat("Input", "Substrate area (mm²):", initialvalue=25.0)
    if diam is None:
        root.destroy()
        return

    paths = filedialog.askopenfilenames(
        title=f"Select ≥2 SQUID .dat files for {diam} µm batch"
    )
    root.destroy()

    if len(paths) < 2:
        print("[!] Please select 2 or more files for averaging.")
        return

    # Ask whether to apply background correction
    do_corr_str = input("Apply background subtraction? (yes/no): ").strip().lower()
    do_corr     = do_corr_str in ('yes', 'y')

    # --- Output folder ---
    batch_folder = os.path.join(OUTDIR_BASE, f"Batch_{diam}um_{ts}")
    os.makedirs(batch_folder, exist_ok=True)

    vol_cm3, mass_g = get_fept_volume_and_mass(area, diam, thick)
    print(f"\nEstimated FePt mass: {mass_g:.3e} g | Volume: {vol_cm3:.3e} cm³")

    # Accumulators — branches kept separate so the averaged mean is a
    # proper closed hysteresis loop (desc averaged with desc, asc with asc)
    h_grid = None
    # Raw emu (top panel — no correction)
    all_raw_h      = []
    all_raw_m_emu  = []
    all_mp_desc, all_mp_asc = [], []   # for mean ± SD of raw panel
    # Corrected + normalised (bottom panel)
    all_raw_m_norm = []
    all_mn_desc, all_mn_asc = [], []   # for mean ± SD of normalised panel
    results_list   = []

    # --- File parsing ---
    for p in paths:
        try:
            with open(p, 'r', encoding='utf-8', errors='ignore') as f:
                lines = f.readlines()
            start_idx = next(
                (i + 1 for i, l in enumerate(lines) if "[Data]" in l), -1
            )
            if start_idx == -1:
                print(f"[!] No [Data] block in {os.path.basename(p)} — skipping.")
                continue

            df = pd.read_csv(p, skiprows=start_idx).dropna(
                subset=["Magnetic Field (Oe)", "Moment (emu)"]
            )
            h = df["Magnetic Field (Oe)"].values
            m = df["Moment (emu)"].values
            h_T = h * OE_TO_T  # Convert to Tesla for plotting

            # Initialise the common field grid (in Tesla) from the first file
            if h_grid is None:
                h_grid = np.linspace(np.min(h_T), np.max(h_T), N_GRID)

            # --- Top panel: raw emu, no correction ---
            # Store in original sweep order to preserve the loop shape for plotting.
            # Split into branches for interpolation so the averaged mean is a
            # proper hysteresis loop, not a collapsed S-curve.
            all_raw_h.append(h_T)
            all_raw_m_emu.append(m)

            (h_d_raw, m_d_raw), (h_a_raw, m_a_raw) = split_branches(h_T, m)
            sort_d = np.argsort(h_d_raw)
            sort_a = np.argsort(h_a_raw)
            all_mp_desc.append(np.interp(h_grid, h_d_raw[sort_d], m_d_raw[sort_d]))
            all_mp_asc.append( np.interp(h_grid, h_a_raw[sort_a], m_a_raw[sort_a]))

            # --- Bottom panel: corrected + normalised M/Msat ---
            if do_corr:
                m_corr, slope = subtract_linear_background(h, m)
                print(f"  [{os.path.basename(p)}] Slope removed: {slope:.4e} emu/Oe")
            else:
                m_corr = m.copy()

            # msat = global max of corrected signal (robust when loop not fully saturated)
            msat = np.max(np.abs(m_corr))
            m_norm = m_corr / msat

            all_raw_m_norm.append(m_norm)

            m_corr_T = m_corr  # field already converted; use h_T branches
            (h_d_c, m_d_c), (h_a_c, m_a_c) = split_branches(h_T, m_norm)
            sort_dc = np.argsort(h_d_c)
            sort_ac = np.argsort(h_a_c)
            all_mn_desc.append(np.interp(h_grid, h_d_c[sort_dc], m_d_c[sort_dc]))
            all_mn_asc.append( np.interp(h_grid, h_a_c[sort_ac], m_a_c[sort_ac]))

            # Metrics computed on corrected loop (h still in Oe for get_metrics)
            hc_t, mr_ms, whyst = get_metrics(h, m_corr, mass_g)
            results_list.append({
                "name": os.path.basename(p),
                "Hc_T": hc_t, "Mr_over_Ms": mr_ms, "Whyst_J_per_kg": whyst
            })

        except Exception as e:
            print(f"[!] Error in {os.path.basename(p)}: {e}")

    if not all_mn_desc:
        print("[!] No files were successfully processed.")
        return

    # --- Statistics: average and SD per branch, per panel ---
    mp_d_avg, mp_d_std = np.mean(all_mp_desc, axis=0), np.std(all_mp_desc, axis=0)
    mp_a_avg, mp_a_std = np.mean(all_mp_asc,  axis=0), np.std(all_mp_asc,  axis=0)
    mn_d_avg, mn_d_std = np.mean(all_mn_desc, axis=0), np.std(all_mn_desc, axis=0)
    mn_a_avg, mn_a_std = np.mean(all_mn_asc,  axis=0), np.std(all_mn_asc,  axis=0)

    # Smooth the SD bands to avoid jagged shading
    sp_d = moving_average(mp_d_std, STD_SMOOTH)
    sp_a = moving_average(mp_a_std, STD_SMOOTH)
    sn_d = moving_average(mn_d_std, STD_SMOOTH)
    sn_a = moving_average(mn_a_std, STD_SMOOTH)

    # ===========================================================================
    # 4. PLOTTING
    # ===========================================================================
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 11))
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

    # --- Top panel: raw emu (no correction) ---
    # Individual curves in original sweep order (loop shape preserved)
    for i, (h_raw, m_raw) in enumerate(zip(all_raw_h, all_raw_m_emu)):
        label = os.path.basename(paths[i])
        ax1.plot(h_raw, m_raw, color=colors[i % len(colors)], lw=1.2, alpha=0.8, label=label)
    # Mean ± SD as a proper closed loop (desc + asc averaged separately)
    ax1.fill_between(h_grid, mp_d_avg - sp_d, mp_d_avg + sp_d, color='black', alpha=0.15)
    ax1.fill_between(h_grid, mp_a_avg - sp_a, mp_a_avg + sp_a, color='black', alpha=0.15,
                     label=r"$\pm$1 SD")
    ax1.plot(h_grid, mp_d_avg, color='black', lw=2, label="Mean")
    ax1.plot(h_grid, mp_a_avg, color='black', lw=2)
    ax1.set_ylabel(r"$m$ (emu)")
    ax1.set_title(
        f"FePt M(H) — {diam} µm batch ({len(paths)} samples)\n{ts}",
        fontweight='bold'
    )
    ax1.legend(loc='lower right', fontsize=7)

    # --- Bottom panel: corrected + normalised M/Msat ---
    for i, (h_raw, m_raw) in enumerate(zip(all_raw_h, all_raw_m_norm)):
        ax2.plot(h_raw, m_raw, color=colors[i % len(colors)], lw=1.2, alpha=0.8)
    ax2.fill_between(h_grid, mn_d_avg - sn_d, mn_d_avg + sn_d, color='black', alpha=0.15)
    ax2.fill_between(h_grid, mn_a_avg - sn_a, mn_a_avg + sn_a, color='black', alpha=0.15,
                     label=r"$\pm$1 SD")
    ax2.plot(h_grid, mn_d_avg, color='black', lw=2, label="Mean")
    ax2.plot(h_grid, mn_a_avg, color='black', lw=2)
    ax2.set_ylabel(r"$M/M_{\mathrm{sat}}$")
    ax2.set_xlabel(r"$\mu_0 H$ (T)")
    ax2.legend(loc='lower right', fontsize=7)

    for ax in [ax1, ax2]:
        ax.axhline(0, color='black', lw=0.5)
        ax.axvline(0, color='black', lw=0.5)
        ax.grid(True, ls=':', alpha=0.6)

    plt.tight_layout()
    plot_base = os.path.join(batch_folder, f"Averaged_Batch_{diam}um_{ts}")
    fig.savefig(plot_base + ".png", dpi=300)
    fig.savefig(plot_base + ".svg")
    plt.close(fig)

    # ===========================================================================
    # 5. REPORT & EXPORTS
    # ===========================================================================

    df_res      = pd.DataFrame(results_list)
    report_path = os.path.join(batch_folder, f"Report_{diam}um_{ts}.txt")
    with open(report_path, "w") as f:
        f.write("SQUID Analysis Report\n")
        f.write(f"Timestamp      : {ts}\n")
        f.write(f"Background corr: {'Yes' if do_corr else 'No'}\n")
        f.write(f"FePt mass      : {mass_g:.6e} g\n\n")
        f.write(df_res.to_string(index=False))
        f.write("\n\nRepeatability (mean ± std):\n")
        for col in ["Hc_T", "Mr_over_Ms", "Whyst_J_per_kg"]:
            avg, std = df_res[col].mean(), df_res[col].std()
            f.write(f"  {col:<20}: {avg:.6f} ± {std:.6f}\n")

    # Averaged data CSV — used as input for the SQUID-Sims overlay script
    df_avg = pd.DataFrame({
        "Field_T":     h_grid,
        "M_norm_desc": mn_d_avg,
        "M_norm_asc":  mn_a_avg,
        "M_emu_desc":  mp_d_avg,
        "M_emu_asc":   mp_a_avg
    })
    avg_csv = os.path.join(batch_folder, f"Averaged_Data_{diam}um_{ts}.csv")
    df_avg.to_csv(avg_csv, index=False)
    print(f"Averaged data CSV saved — use as input for the SQUID-Sims overlay script.")
    print(f"\nDONE. All outputs saved in: {batch_folder}")


if __name__ == "__main__":
    main()
